# 10 — Final Analysis

**PLTMH–ELC–QLSTM**

Notebook ini merupakan tahap analisis akhir setelah seluruh model dan
eksperimen *closed-loop* dibekukan.

Ruang lingkup:

1. audit hasil primer;
2. interpretasi ilmiah;
3. hubungan akurasi estimasi gain dengan kinerja pengendalian;
4. audit beban komputasi;
5. audit usaha kendali;
6. pembekuan klaim ilmiah;
7. verifikasi paket tabel dan gambar tesis;
8. verifikasi draft Bab IV.

Notebook ini **tidak membuka kembali tahap model selection atau
closed-loop tuning**.


## 1. Scientific Freeze

Kesimpulan primer dibatasi pada dua keluarga dinamik uji independen
yang telah dipraregistrasikan.

Hasil beku:

- Fixed PI mempunyai rerata RMSE primer terendah;
- LSTM–PI bersifat bergantung pada skenario;
- QLSTM–PI tidak menunjukkan keunggulan terhadap Fixed PI maupun
  LSTM–PI;
- tidak ada klaim signifikansi statistik;
- tidak ada klaim generalisasi independen untuk G3;
- tidak ada *quantum advantage*;
- hasil negatif QLSTM dipertahankan.

Tidak diperbolehkan mengubah model, scheduler, skenario, simulator,
atau metric definition untuk memperbaiki hasil yang telah diketahui.


In [ ]:
# ============================================================
# 10.1 — PROJECT + FROZEN FINAL-EVIDENCE INTEGRITY
# ============================================================

from pathlib import Path

import hashlib
import json
import sys

import numpy as np
import pandas as pd


EXPECTED_UPSTREAM_CHECKPOINT = "2dc597f785fce3ecc0ddffa5c3fd3f19ee0c56dc"

EXPECTED_NOTEBOOK_09_SHA256 = (
    "1a882fd7698990826f571b44bc053e0e86d9125116c6c316a1ae2695824382ff"
)

EXPECTED_RUNTIME_SOURCE_SNAPSHOT_SHA256 = (
    "259774a1296d9723fbb451e3b2e9d68ef7f8c106ce7c248b2f5ed370ad58f043"
)

EXPECTED_FINAL_ANALYSIS_ARTIFACT_SHA256 = (
    {
    "data/closed_loop/cell156_computational_burden_audit.csv": "81ba7ff0adafc7d4309dd193c9136f3d3d28c0ee5a795937669e86dee8c944ed",
    "data/closed_loop/cell156_model_control_link_audit.csv": "5ae2a03ebf83ea6785af9def0fa84c90374c83cf8157934c1b4444305e1c68b0",
    "data/closed_loop/cell156_peak_rocof_timing_audit.csv": "e9299857922ea629a1fd3d35c2b9ec0aba1eb8103a892eb8430c55f3cfccbe70",
    "data/closed_loop/cell156_primary_family_effect_audit.csv": "5a78a87578a5d4fb166ae9c5b53ba7085db1874b73a8e787b03655d0ce3182f6",
    "data/closed_loop/cell156_scheduler_control_effort_audit.csv": "cfd2747e7a58e3c2065521c276536ad4000be9db34bd7748097626d93c6fc75e",
    "data/closed_loop/cell156_scientific_claim_freeze.csv": "4b9b0c612998ba9b944c21f4feb64ef7bec0a9807d0b2a73bcdf7bda62170581",
    "data/recovery/cell156_closed_loop_interpretation_freeze.json": "650a3ad9c9ba0bf32b3f7cce3b6bf254202406860f3850fe270f4d50e2dbf41d",
    "data/recovery/cell156_closed_loop_resume_notes.txt": "e826208042df6a063171cdd5ab2b2363866692df71a66c8dc6ff18d616e8c08e",
    "reports/thesis_results/cell157_figure_captions.csv": "437c90d844f80d0b7cca0f0131aed2f1a025ae53471232b5e4836d76ce5f5c35",
    "reports/thesis_results/cell157_thesis_evidence_manifest.json": "424d99e5ad57aa409f349ba598ff0785f8e543860d001b097358d5835883a0df",
    "reports/thesis_results/cell157_thesis_results_summary.md": "4f99b38505a78c42a11fbba90406e25491fdf3f6cb3ba5f00902428cf5b12993",
    "reports/thesis_results/chapter_iv/Bab_IV_Hasil_dan_Pembahasan_draft.md": "edd37a5221893a126ba71b313d1bf0c483897ba242f5d4c14278ee7bb2dadc3b",
    "reports/thesis_results/chapter_iv/cell158_bab_iv_evidence_map.csv": "85d160ac642c6a2f23cb8e1bc2d4de424d99b07d317f143df899f461fcfa4026",
    "reports/thesis_results/chapter_iv/cell159_review_sections_4_1_to_4_3.md": "5b8c64747885d5c00bd1f0647ff9741b14042ed039547ae5bdffc85abbcb4511",
    "reports/thesis_results/figures/fig157_cl01_applied_gain_trajectories.png": "7ec96f353ce6684dd798dfb81e332c88a2d3b7105b742a04d60f5881afefbd12",
    "reports/thesis_results/figures/fig157_cl01_dump_duty_response.png": "52b6ce377c621be4f5ce2f408dfa5c3e451608797af34bccf0c4f1b1708a77e1",
    "reports/thesis_results/figures/fig157_cl01_frequency_response.png": "b411f95fcae6eaa85e37d56895fb77a85ba584ff80dee3c18555e32fdc669215",
    "reports/thesis_results/figures/fig157_cl02_applied_gain_trajectories.png": "7fb5e878e00f8b27006187e1c79a6923a13eb88a11b5ffa5753158a3b7cae8f8",
    "reports/thesis_results/figures/fig157_cl02_dump_duty_response.png": "a1a8e741092112e7c253d10b41def2f754a2fd9addcb60bf98128599e52f94f7",
    "reports/thesis_results/figures/fig157_cl02_frequency_response.png": "d1377e5144b425c5c9612441457e693880fc8f86b75656f1c0a99fb64259cd70",
    "reports/thesis_results/figures/fig157_computational_runtime_comparison.png": "8e1840e1d8f65dab493d4e8d8584eb613f635c08726ef5872a4db3e99d127c6f",
    "reports/thesis_results/figures/fig157_primary_rmse_comparison.png": "4b762acea8e1816ae8a5dc94a37886f07189610c282f1c4f5d285b6c7cf6218f",
    "reports/thesis_results/tables/table157_computational_burden.csv": "87c9c458ee9e8ee858bae076c82aee3c81f6cfc3f9ba9d910412cf277b7ff339",
    "reports/thesis_results/tables/table157_primary_descriptive_summary.csv": "953166f449359c9603b6261e31ce456ac9498ab4123519bf5e7d67baf0947d9d",
    "reports/thesis_results/tables/table157_primary_heldout_controller_comparison.csv": "81f2ff11310aa11c035a23feea5eb86ec6d1d4d1aa6516de85c39aa28439e832"
}
)

CRITICAL_RELPATHS = (
    {
    "bab_iv_draft": "reports/thesis_results/chapter_iv/Bab_IV_Hasil_dan_Pembahasan_draft.md",
    "bab_iv_evidence_map": "reports/thesis_results/chapter_iv/cell158_bab_iv_evidence_map.csv",
    "bab_iv_review": "reports/thesis_results/chapter_iv/cell159_review_sections_4_1_to_4_3.md",
    "claims": "data/closed_loop/cell156_scientific_claim_freeze.csv",
    "closed_loop_metrics": "data/closed_loop/cell155_closed_loop_metrics.csv",
    "computational_burden": "data/closed_loop/cell156_computational_burden_audit.csv",
    "figure_captions": "reports/thesis_results/cell157_figure_captions.csv",
    "interpretation_freeze": "data/recovery/cell156_closed_loop_interpretation_freeze.json",
    "model_control_link": "data/closed_loop/cell156_model_control_link_audit.csv",
    "model_test_comparison": "models/qlstm/cell149_frozen_model_test_comparison.csv",
    "peak_rocof_timing": "data/closed_loop/cell156_peak_rocof_timing_audit.csv",
    "primary_family_effect": "data/closed_loop/cell156_primary_family_effect_audit.csv",
    "primary_summary": "data/closed_loop/cell155_primary_family_summary.csv",
    "scheduler_control_effort": "data/closed_loop/cell156_scheduler_control_effort_audit.csv",
    "thesis_manifest": "reports/thesis_results/cell157_thesis_evidence_manifest.json",
    "thesis_summary": "reports/thesis_results/cell157_thesis_results_summary.md"
}
)


def sha256_file(path):

    digest = hashlib.sha256()

    with open(path, "rb") as f:

        for chunk in iter(
            lambda: f.read(1024 * 1024),
            b"",
        ):

            digest.update(chunk)

    return digest.hexdigest()


candidate_roots = [
    Path("/content/PLTMH-ELC-QLSTM"),
    Path.cwd(),
    Path.cwd().parent,
]


PROJECT_ROOT = None


for candidate in candidate_roots:

    candidate = candidate.resolve()

    if (
        (candidate / ".git").exists()
        and
        (candidate / "notebooks").exists()
        and
        (candidate / "reports").exists()
    ):

        PROJECT_ROOT = candidate
        break


if PROJECT_ROOT is None:

    raise RuntimeError(
        "PLTMH-ELC-QLSTM repository root not found."
    )


if str(PROJECT_ROOT) not in sys.path:

    sys.path.insert(
        0,
        str(PROJECT_ROOT),
    )


NOTEBOOK_09_PATH = (
    PROJECT_ROOT
    /
    "notebooks"
    /
    "09_closed_loop.ipynb"
)


if (
    sha256_file(
        NOTEBOOK_09_PATH
    )
    !=
    EXPECTED_NOTEBOOK_09_SHA256
):

    raise RuntimeError(
        "Upstream Notebook 09 changed."
    )


RUNTIME_SOURCE_SNAPSHOT_PATH = (
    PROJECT_ROOT
    /
    "data"
    /
    "recovery"
    /
    "cell167_final_analysis_runtime_source_snapshot.json"
)


if (
    sha256_file(
        RUNTIME_SOURCE_SNAPSHOT_PATH
    )
    !=
    EXPECTED_RUNTIME_SOURCE_SNAPSHOT_SHA256
):

    raise RuntimeError(
        "Final-analysis source archive changed."
    )


artifact_integrity = {}


for relative_path, expected_sha in (
    EXPECTED_FINAL_ANALYSIS_ARTIFACT_SHA256.items()
):

    path = (
        PROJECT_ROOT
        /
        relative_path
    )


    artifact_integrity[
        relative_path
    ] = bool(
        path.exists()
        and
        sha256_file(
            path
        )
        ==
        expected_sha
    )


FROZEN_FINAL_ANALYSIS_ARTIFACTS_VALID = all(
    artifact_integrity.values()
)


print(
    "Frozen final-analysis artifacts:",
    len(
        artifact_integrity
    )
)

print(
    "All frozen artifacts valid:",
    FROZEN_FINAL_ANALYSIS_ARTIFACTS_VALID
)


if not FROZEN_FINAL_ANALYSIS_ARTIFACTS_VALID:

    failed = [
        path
        for path, status
        in artifact_integrity.items()
        if not status
    ]

    print(
        "Failed artifacts:",
        failed
    )

    raise RuntimeError(
        "Frozen final-analysis evidence changed."
    )


In [ ]:
# ============================================================
# 10.2 — SCIENTIFIC CONCLUSION FREEZE
# ============================================================

EXPECTED_INTERPRETATION = {
    "cell155_raw_result_checkpoint": "bb5afb3a09041c292d914eb10bde0b495db1e7a7",
    "computational_burden": {
        "caveat": "Paired overhead is not isolated pure model inference latency. It includes implementation-level simulation overhead.",
        "fixed_mean_full_run_runtime_s": 0.6747233790003975,
        "gain_update_budget_s": 0.1,
        "lstm_approx_overhead_per_update_s": 0.0029975172162107194,
        "lstm_current_10hz_budget_demonstrated": true,
        "lstm_mean_full_run_runtime_s": 0.8965396529999907,
        "qlstm_approx_overhead_per_update_s": 0.43102371570269593,
        "qlstm_current_10hz_budget_demonstrated": false,
        "qlstm_mean_full_run_runtime_s": 32.5704783409999,
        "qlstm_to_lstm_runtime_ratio": 36.32909959086911
    },
    "control_effort": {
        "fallback_used": false,
        "gain_guard_materially_active": true,
        "qlstm_duty_tv_ratio_vs_fixed": 6.404012444456979,
        "qlstm_duty_tv_ratio_vs_lstm": 5.5674973577030435
    },
    "model_control_link": {
        "lstm_gain_test_mse": 0.86862868,
        "proportional_mapping_claim": false,
        "qlstm_closed_loop_mean_rmse_increase_vs_lstm_pct": 16.945621886292017,
        "qlstm_gain_mse_increase_vs_lstm_pct": 241.17563465189141,
        "qlstm_gain_test_mse": 2.963549411758347
    },
    "next_stage": "OPTIONAL_PREDECLARED_ROBUSTNESS_OR_FINAL_THESIS_SYNTHESIS",
    "post_result_policy": {
        "model_retraining_allowed": false,
        "primary_result_replacement_allowed": false,
        "scenario_selection_change_allowed": false,
        "scheduler_retuning_allowed": false,
        "simulator_modification_to_improve_result_allowed": false
    },
    "primary_evidence": {
        "fixed_pi_mean_rmse_hz": 0.012478396116523651,
        "fixed_primary_wins": 1,
        "independent_family_count": 2,
        "lstm_mean_rmse_increase_vs_fixed_pct": 10.670851634471155,
        "lstm_pi_mean_rmse_hz": 0.0138099472524795,
        "lstm_primary_wins": 1,
        "qlstm_mean_rmse_increase_vs_fixed_pct": 29.424715690787867,
        "qlstm_mean_rmse_increase_vs_lstm_pct": 16.945621886292017,
        "qlstm_pi_mean_rmse_hz": 0.01615012869658105,
        "qlstm_primary_wins": 0
    },
    "scientific_conclusion": {
        "lstm_consistent_superiority_over_fixed_supported": false,
        "negative_result_retained": true,
        "primary": "Pada dua keluarga dinamik uji independen yang telah dipraregistrasikan, QLSTM\u2013PI tidak menunjukkan keunggulan kinerja closed-loop terhadap PI gain tetap maupun LSTM\u2013PI berdasarkan RMSE deviasi frekuensi. PI gain tetap menghasilkan rerata RMSE terendah, sedangkan manfaat LSTM\u2013PI bersifat bergantung pada skenario: meningkat pada D20_L+10 tetapi menurun pada D40_L-20.",
        "qlstm": "QLSTM tetap mempertahankan kestabilan pada seluruh skenario yang diuji, tetapi tidak memenangkan RMSE pada kedua keluarga uji utama. Hasil tersebut disertai variasi duty dump-load dan beban komputasi yang lebih besar pada implementasi simulator kuantum saat ini.",
        "qlstm_superiority_supported": false,
        "scope": "Kesimpulan dibatasi pada dua keluarga uji independen, model ELC averaged, daya mekanik konstan, simulasi deterministik, serta kebijakan observasi 20 Hz dan pembaruan gain 10 Hz yang telah dibekukan. Hasil tidak mendukung klaim generalisasi independen untuk G3 dan tidak digunakan untuk membuat klaim signifikansi statistik.",
        "statistical_significance_claim_allowed": false
    },
    "secondary_evidence": {
        "cl03_role": "BOUNDARY_STRESS_ONLY",
        "cl04_role": "G3_IN_DISTRIBUTION_DIAGNOSTIC_ONLY",
        "independent_g3_generalization": false
    },
    "stage": "CLOSED_LOOP_SCIENTIFIC_INTERPRETATION_FREEZE",
    "timing": {
        "first_adaptive_update_s": 2.6,
        "peak_identical_across_primary_controllers": true,
        "primary_peak_before_or_at_first_update": true,
        "primary_rocof_before_or_at_first_update": true,
        "rocof_identical_across_primary_controllers": true
    }
}


INTERPRETATION_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "interpretation_freeze"
    ]
)


CLAIMS_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "claims"
    ]
)


interpretation = json.loads(
    INTERPRETATION_PATH.read_text(
        encoding="utf-8"
    )
)


claims = pd.read_csv(
    CLAIMS_PATH
)


INTERPRETATION_EXACT = bool(
    interpretation
    ==
    EXPECTED_INTERPRETATION
)


if not INTERPRETATION_EXACT:

    raise RuntimeError(
        "Frozen scientific interpretation changed."
    )


print(
    "Scientific conclusion:"
)

print(
    interpretation[
        "scientific_conclusion"
    ][
        "primary"
    ]
)


print(
    "\nScope:"
)

print(
    interpretation[
        "scientific_conclusion"
    ][
        "scope"
    ]
)


print(
    "\nQLSTM superiority supported:",
    interpretation[
        "scientific_conclusion"
    ][
        "qlstm_superiority_supported"
    ]
)

print(
    "Negative result retained:",
    interpretation[
        "scientific_conclusion"
    ][
        "negative_result_retained"
    ]
)

print(
    "Statistical significance allowed:",
    interpretation[
        "scientific_conclusion"
    ][
        "statistical_significance_claim_allowed"
    ]
)

print(
    "Independent G3 generalization:",
    interpretation[
        "secondary_evidence"
    ][
        "independent_g3_generalization"
    ]
)


print(
    "\nScientific-claim table:"
)


print(
    claims[
        [
            "claim_id",
            "status",
            "claim",
        ]
    ].to_string(
        index=False
    )
)


QLSTM_SUPERIORITY_SUPPORTED = False

STATISTICAL_SIGNIFICANCE_CLAIM_ALLOWED = False

INDEPENDENT_G3_CLAIM_ALLOWED = False

POST_RESULT_RETUNING_ALLOWED = False


In [ ]:
# ====================================================
# 10.3 — PRIMARY CONTROL RESULTS
# ====================================================

PRIMARY_SUMMARY_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "primary_summary"
    ]
)


primary_summary = pd.read_csv(
    PRIMARY_SUMMARY_PATH
)


expected_controllers = {
    "FIXED_PI",
    "LSTM_PI",
    "QLSTM_PI",
}


if set(
    primary_summary[
        "controller"
    ]
) != expected_controllers:

    raise RuntimeError(
        "Primary controller set changed."
    )


print(
    primary_summary.to_string(
        index=False
    )
)


fixed_rmse = float(
    primary_summary.loc[
        primary_summary[
            "controller"
        ]
        ==
        "FIXED_PI",
        "mean_primary_rmse_hz",
    ].iloc[0]
)


lstm_rmse = float(
    primary_summary.loc[
        primary_summary[
            "controller"
        ]
        ==
        "LSTM_PI",
        "mean_primary_rmse_hz",
    ].iloc[0]
)


qlstm_rmse = float(
    primary_summary.loc[
        primary_summary[
            "controller"
        ]
        ==
        "QLSTM_PI",
        "mean_primary_rmse_hz",
    ].iloc[0]
)


ranking = (
    primary_summary
    .sort_values(
        "rank_by_mean_primary_rmse"
    )[
        "controller"
    ]
    .tolist()
)


if ranking != [
    "FIXED_PI",
    "LSTM_PI",
    "QLSTM_PI",
]:

    raise RuntimeError(
        "Primary descriptive ranking changed."
    )


print(
    "\nFixed PI mean primary RMSE [Hz]:",
    fixed_rmse
)

print(
    "LSTM-PI mean primary RMSE [Hz]:",
    lstm_rmse
)

print(
    "QLSTM-PI mean primary RMSE [Hz]:",
    qlstm_rmse
)

print(
    "Descriptive ranking:",
    ranking
)

print(
    "\nIndependent family count: 2"
)

print(
    "Interpretation: descriptive, not inferential."
)


In [ ]:
# ====================================================
# 10.4 — MODEL/CONTROL LINK + COMPUTATIONAL BURDEN
# ====================================================

MODEL_TEST_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "model_test_comparison"
    ]
)


COMPUTE_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "computational_burden"
    ]
)


MODEL_CONTROL_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "model_control_link"
    ]
)


CONTROL_EFFORT_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "scheduler_control_effort"
    ]
)


model_test = pd.read_csv(
    MODEL_TEST_PATH
)


computational = pd.read_csv(
    COMPUTE_PATH
)


model_control = pd.read_csv(
    MODEL_CONTROL_PATH
)


control_effort = pd.read_csv(
    CONTROL_EFFORT_PATH
)


print(
    "Frozen model-stage test comparison:"
)


print(
    model_test.to_string(
        index=False
    )
)


print(
    "\nComputational burden audit:"
)


print(
    computational.to_string(
        index=False
    )
)


print(
    "\nModel/control link audit:"
)


print(
    model_control.to_string(
        index=False
    )
)


print(
    "\nScheduler/control-effort audit:"
)


print(
    control_effort.to_string(
        index=False
    )
)


print(
    "\nInterpretation:"
)

print(
    "Gain-regression error and closed-loop "
    "frequency-control degradation are not "
    "treated as proportional quantities."
)

print(
    "Current CPU/PennyLane timing is implementation-level "
    "evidence and is not a quantum-speedup benchmark."
)


QUANTUM_SPEEDUP_CLAIM_ALLOWED = False


In [ ]:
# ====================================================
# 10.5 — THESIS TABLE / FIGURE PACKAGE AUDIT
# ====================================================

THESIS_MANIFEST_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "thesis_manifest"
    ]
)


thesis_manifest = json.loads(
    THESIS_MANIFEST_PATH.read_text(
        encoding="utf-8"
    )
)


package_paths = (
    thesis_manifest[
        "tables"
    ]
    +
    thesis_manifest[
        "figures"
    ]
    +
    [
        thesis_manifest[
            "summary"
        ],
        thesis_manifest[
            "figure_captions"
        ],
    ]
)


package_status = []


for relative_path in package_paths:

    path = (
        PROJECT_ROOT
        /
        relative_path
    )


    package_status.append(
        {
            "path":
                relative_path,

            "exists":
                path.exists(),

            "size_bytes":
                (
                    path.stat().st_size
                    if path.exists()
                    else 0
                ),
        }
    )


package_status = pd.DataFrame(
    package_status
)


PACKAGE_READY = bool(
    package_status[
        "exists"
    ].all()

    and

    (
        package_status[
            "size_bytes"
        ]
        >
        0
    ).all()
)


print(
    package_status.to_string(
        index=False
    )
)


print(
    "\nTables:",
    len(
        thesis_manifest[
            "tables"
        ]
    )
)

print(
    "Figures:",
    len(
        thesis_manifest[
            "figures"
        ]
    )
)

print(
    "Package ready:",
    PACKAGE_READY
)


if not PACKAGE_READY:

    raise RuntimeError(
        "Thesis evidence package incomplete."
    )


In [ ]:
# ====================================================
# 10.6 — BAB IV DRAFT + REVIEW AUDIT
# ====================================================

BAB_IV_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "bab_iv_draft"
    ]
)


BAB_IV_REVIEW_PATH = (
    PROJECT_ROOT
    /
    CRITICAL_RELPATHS[
        "bab_iv_review"
    ]
)


bab_iv = BAB_IV_PATH.read_text(
    encoding="utf-8"
)


bab_iv_review = (
    BAB_IV_REVIEW_PATH.read_text(
        encoding="utf-8"
    )
)


required_sections = [
    f"## 4.{i}"
    for i in range(
        1,
        11,
    )
]


BAB_IV_STRUCTURE_VALID = all(
    section
    in
    bab_iv

    for section
    in required_sections
)


BAB_IV_4_1_TO_4_3_REVIEWED = all(
    phrase
    in
    bab_iv_review

    for phrase
    in (
        "## Status",
        "PASS",
        "no statistical-significance claim",
        "no independent G3 generalization claim",
        "no post-result model/scheduler tuning",
        "no quantum-advantage claim",
        "negative QLSTM result retained",
    )
)


print(
    "Draft characters:",
    len(
        bab_iv
    )
)

print(
    "Sections 4.1–4.10 present:",
    BAB_IV_STRUCTURE_VALID
)

print(
    "Sections 4.1–4.3 reviewed:",
    BAB_IV_4_1_TO_4_3_REVIEWED
)


if not (
    BAB_IV_STRUCTURE_VALID
    and
    BAB_IV_4_1_TO_4_3_REVIEWED
):

    raise RuntimeError(
        "Bab IV draft/review contract failed."
    )


In [ ]:
# ====================================================
# 10.7 — FINAL-ANALYSIS SOURCE ARCHIVE AUDIT
# ====================================================

snapshot = json.loads(
    RUNTIME_SOURCE_SNAPSHOT_PATH.read_text(
        encoding="utf-8"
    )
)


expected_labels = [
    "156",
    "157",
    "158",
    "159",
    "159-FIX",
]


SOURCE_ARCHIVE_COMPLETE = bool(
    snapshot[
        "labels"
    ]
    ==
    expected_labels

    and

    all(
        label
        in
        snapshot[
            "cells"
        ]

        for label
        in
        expected_labels
    )

    and

    snapshot[
        "authoritative_cells"
    ]
    ==
    [
        "156",
        "157",
        "158",
        "159-FIX",
    ]
)


print(
    "Archived source cells:"
)


for label in expected_labels:

    record = (
        snapshot[
            "cells"
        ][
            label
        ]
    )


    print(
        f"CELL {label:7s} | "
        f"{record['source_chars']:7d} chars | "
        f"{record['source_sha256']}"
    )


print(
    "\nCELL 159 authoritative:",
    False
)

print(
    "CELL 159-FIX authoritative:",
    True
)

print(
    "SOURCE_ARCHIVE_COMPLETE:",
    SOURCE_ARCHIVE_COMPLETE
)


if not SOURCE_ARCHIVE_COMPLETE:

    raise RuntimeError(
        "Final-analysis source archive incomplete."
    )


## 2. Reproduction dan Regeneration Policy

Hasil primer, interpretasi, tabel, gambar, dan draft Bab IV telah
disusun dari evidence yang dibekukan.

Mode default notebook ini adalah **verifikasi**.

Tidak dilakukan:

- simulasi ulang;
- inferensi model ulang;
- training ulang;
- penggantian hasil primer;
- perubahan interpretasi setelah hasil diketahui.

Reproduksi penuh hanya boleh dilakukan sebagai workflow terpisah
dengan protokol ilmiah yang sama.


In [ ]:
# ====================================================
# 10.8 — EXPLICIT REGENERATION GUARD
# ====================================================

RERUN_FINAL_ANALYSIS_PIPELINE = False

REGENERATE_THESIS_PACKAGE = False

REWRITE_BAB_IV = False


SIMULATION_RERUN_PERFORMED = False

MODEL_INFERENCE_PERFORMED = False

MODEL_TRAINING_PERFORMED = False

PRIMARY_RESULT_REPLACED = False


if not RERUN_FINAL_ANALYSIS_PIPELINE:

    print(
        "RERUN_FINAL_ANALYSIS_PIPELINE = False"
    )


if not REGENERATE_THESIS_PACKAGE:

    print(
        "REGENERATE_THESIS_PACKAGE = False"
    )


if not REWRITE_BAB_IV:

    print(
        "REWRITE_BAB_IV = False"
    )


print(
    "Frozen Cell-156–159-FIX evidence "
    "remains authoritative."
)


## 3. Status Akhir Pipeline Notebook

Urutan notebook penelitian:

`05_generate_dataset.ipynb`

→ `06_preprocessing.ipynb`

→ `07_train_lstm.ipynb`

→ `08_train_qlstm.ipynb`

→ `09_closed_loop.ipynb`

→ `10_final_analysis.ipynb`

Seluruh notebook telah dipisahkan berdasarkan tahap ilmiahnya dan
tidak lagi bergantung pada variabel global dari notebook Colab panjang
yang digunakan selama pengembangan.

Langkah berikutnya adalah **cross-notebook integrity audit**, bukan
menjalankan ulang eksperimen.


In [ ]:
# ====================================================
# 10.9 — FINAL ANALYSIS SUMMARY
# ====================================================

NOTEBOOK_10_FINAL_ANALYSIS_READY = all(
    [
        FROZEN_FINAL_ANALYSIS_ARTIFACTS_VALID,
        INTERPRETATION_EXACT,
        PACKAGE_READY,
        BAB_IV_STRUCTURE_VALID,
        BAB_IV_4_1_TO_4_3_REVIEWED,
        SOURCE_ARCHIVE_COMPLETE,
        not QLSTM_SUPERIORITY_SUPPORTED,
        not STATISTICAL_SIGNIFICANCE_CLAIM_ALLOWED,
        not INDEPENDENT_G3_CLAIM_ALLOWED,
        not POST_RESULT_RETUNING_ALLOWED,
        not QUANTUM_SPEEDUP_CLAIM_ALLOWED,
        not SIMULATION_RERUN_PERFORMED,
        not MODEL_INFERENCE_PERFORMED,
        not MODEL_TRAINING_PERFORMED,
        not PRIMARY_RESULT_REPLACED,
    ]
)


print("=" * 72)
print("10_final_analysis.ipynb — SUMMARY")
print("=" * 72)


print(
    "Independent primary families       : 2"
)

print(
    "Fixed PI mean primary RMSE [Hz]    :",
    fixed_rmse
)

print(
    "LSTM-PI mean primary RMSE [Hz]     :",
    lstm_rmse
)

print(
    "QLSTM-PI mean primary RMSE [Hz]    :",
    qlstm_rmse
)

print(
    "QLSTM superiority supported        : False"
)

print(
    "Statistical significance claim     : False"
)

print(
    "Independent G3 claim               : False"
)

print(
    "Quantum advantage claim            : False"
)

print(
    "Post-result tuning                 : False"
)

print(
    "Thesis tables                      :",
    len(
        thesis_manifest[
            "tables"
        ]
    )
)

print(
    "Thesis figures                     :",
    len(
        thesis_manifest[
            "figures"
        ]
    )
)

print(
    "Bab IV sections 4.1–4.10           :",
    BAB_IV_STRUCTURE_VALID
)

print(
    "Bab IV 4.1–4.3 reviewed            :",
    BAB_IV_4_1_TO_4_3_REVIEWED
)

print(
    "Final-analysis source archived     :",
    SOURCE_ARCHIVE_COMPLETE
)

print(
    "Simulation rerun                   : False"
)

print(
    "Model inference                    : False"
)

print(
    "Model training                     : False"
)

print(
    "Primary result replaced            : False"
)

print(
    "NOTEBOOK 10 FINAL ANALYSIS READY   :",
    NOTEBOOK_10_FINAL_ANALYSIS_READY
)
